# Portfolio：两条策略的 PnL、等权组合与 convex_sr 组合

这份 notebook 展示一个完整的策略组合流程。我们不直接跳到最终结果，而是把每一步拆开：

1. 读取 K 线数据；
2. 从 `stra_assets.get_all_strategy_exprs()` 里选两条策略；
3. 把策略信号转成持仓；
4. 用 `bt.price` 生成逐行 PnL；
5. 按日期汇总成日 PnL；
6. 分别画两条策略的累计 PnL；
7. 画等权组合曲线；
8. 用 `opt.convex_sr(lower_bound=0.0, up_bound=1.0)` 求权重；
9. 画 convex_sr 加权后的组合曲线。

这里的重点是“组合逻辑怎么组织”，不是证明某个策略未来有效。`convex_sr` 用的是样本内统计，真实研究必须做训练/验证切分、滚动训练和交易成本约束。


In [ ]:
import os
import sys

LOCAL_QUST_SOURCE = "/root/otters/otters-py/python"
if os.path.isdir(LOCAL_QUST_SOURCE) and LOCAL_QUST_SOURCE not in sys.path:
    sys.path.insert(0, LOCAL_QUST_SOURCE)

import qust as qs
import qust.future.future  # 注册 future / stra / bt / opt / monitor 等命名空间
from qust import col, stra_assets
from qust._polars import pl


pl.Config.set_tbl_rows(18)
pl.Config.set_tbl_cols(24)

KLINE_PATH = "/root/qust-py/examples/data/data_kline2.parquet"


## 1. 读取数据和策略库

`data_kline2.parquet` 是本地样例 K 线数据，包含多个期货品种。内置策略通过 `stra_assets.get_all_strategy_exprs()` 获取，每个策略表达式的输入通常是 `datetime/open/high/low/close/volume`，输出是四列布尔信号：

| 输出列 | 含义 |
| --- | --- |
| `open_long_sig` | 开多信号 |
| `exit_long_sig` | 平多信号 |
| `open_short_sig` | 开空信号 |
| `exit_short_sig` | 平空信号 |

后面我们会把这四列信号送入 `stra.to_hold_two_sides()`，生成目标持仓。


In [53]:
data_kline = pl.read_parquet(KLINE_PATH)
stras = stra_assets.get_all_strategy_exprs()

selected_indices = [0, 3]
selected = [(idx, stras[idx]) for idx in selected_indices]
strategy_names = [expr.get_metadata_value("name") or f"strategy_{idx}" for idx, expr in selected]

strategy_info = pl.DataFrame(
    {
        "index": selected_indices,
        "name": strategy_names,
        "description": [expr.get_metadata().get("description", "") for _, expr in selected],
    }
)

print("data shape:", data_kline.shape)
print("strategy count:", len(stras))
strategy_info


data shape: (610463, 8)
strategy count: 130


index,name,description
i64,str,str
0,"""c02""","""C02 策略 输入列：datetime, open, hig…"
3,"""c05""","""C05 策略 输入列：datetime, open, hig…"


## 2. 从策略信号到日 PnL

组合之前必须先统一策略输出的口径。这里每条策略都走同一条回测管线：

```text
K 线数据
-> 内置策略表达式，输出 open/exit 信号
-> to_hold_two_sides，四列开平仓信号变成持仓 hold
-> bt.price，按 close 与 hold 计算逐行 pnl
-> over("ticker")，每个品种独立维护持仓和回测状态
-> group_by(date)，把所有品种的逐行 pnl 汇总成日 pnl
```

注意 `over("ticker")` 放在回测链路外层，是为了让每个品种独立维护持仓状态；最后再按日期把所有品种的 PnL 汇总成一条策略曲线。


In [54]:
def strategy_daily_pnl_expr(stra_expr, alias: str):
    return (
        col
        .with_cols(stra_expr)
        .with_cols(
            col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra
            .to_hold_two_sides()
            .expanding()
            .alias("hold")
        )
        .with_cols(
            col("close", "hold")
            .bt
            .price(fee_rate=0.0)
            .expanding()
        )
        .over("ticker")
        .select(
            col("pnl")
            .sum()
            .alias(alias)
            .group_by(col("datetime").dt.date().alias("date"))
        )
    )


pnl_tables = []
for (idx, stra_expr), name in zip(selected, strategy_names):
    daily_pnl = strategy_daily_pnl_expr(stra_expr, name).calc_data(data_kline).sort("date")
    pnl_tables.append(daily_pnl)
    print(f"{name}: rows={daily_pnl.height}, total_pnl={daily_pnl[name].sum():.4f}")

portfolio_daily = pnl_tables[0]
for table in pnl_tables[1:]:
    portfolio_daily = portfolio_daily.join(table, on="date", how="inner")
portfolio_daily = portfolio_daily.fill_null(0.0).sort("date")

portfolio_daily.head(10)


c02: rows=461, total_pnl=-587.3199


c05: rows=461, total_pnl=2165.0399


date,c02,c05
date,f64,f64
2022-07-02,0.0,0.26001
2022-07-04,-26.940002,-77.880005
2022-07-05,-23.0,288.28009
2022-07-06,-0.23999,160.300018
2022-07-07,36.0,-18.02005
2022-07-08,-11.0,-41.919952
2022-07-09,0.0,-1.000061
2022-07-11,-8.0,-14.659912
2022-07-12,-37.119995,507.999969


## 3. 分别画两条策略的累计 PnL

日 PnL 本身会正负跳动，不适合直接观察策略整体表现，所以先对每条策略做累计和。这里仍然用 qust 表达式完成累计：

```python
col("pnl").sum().expanding()
```

`expanding()` 表示从第一天累积到当前天。


In [55]:
strategy_curve_expr = col(
    "date",
    *[col(name).sum().expanding().alias(name) for name in strategy_names],
)

strategy_curves = strategy_curve_expr.calc_data(portfolio_daily)
strategy_curves.tail(8)


date,c02,c05
date,f64,f64
2024-01-24,-563.319885,2097.259888
2024-01-25,-563.319885,2113.359924
2024-01-26,-571.319885,2136.039917
2024-01-27,-571.319885,2135.899902
2024-01-29,-571.319885,2177.519897
2024-01-30,-574.319885,2227.759857
2024-01-31,-574.319885,2155.599884
2024-02-01,-587.319885,2165.039886


In [56]:
strategy_0_runtime = col("date", strategy_names[0]).monitor(
    f"{strategy_names[0]}_cum_pnl",
    show_axis_label=True,
).line().runtime()

strategy_0_runtime.plot(strategy_curves.select("date", strategy_names[0]), open_in_jupyter=True, auto_open=False, height=420)


In [57]:
strategy_1_runtime = col("date", strategy_names[1]).monitor(
    f"{strategy_names[1]}_cum_pnl",
    show_axis_label=True,
).line().runtime()

strategy_1_runtime.plot(strategy_curves.select("date", strategy_names[1]), open_in_jupyter=True, auto_open=False, height=420)


## 4. 等权组合

最简单的组合方式是等权：两条策略每天的 PnL 各占 50%。如果策略之间相关性不高，等权组合常常能降低曲线波动；如果两条策略高度同向，等权的分散效果就会很弱。

下面同时画出两条单策略累计曲线和等权组合累计曲线。


In [58]:
equal_weight_expr = (
    (col(strategy_names[0]) + col(strategy_names[1])) / col.lit(2.0)
).alias("equal_weight")

portfolio_curve_expr = col(
    "date",
    *[col(name).sum().expanding().alias(name) for name in strategy_names],
    equal_weight_expr.sum().expanding().alias("equal_weight"),
)

portfolio_curves = portfolio_curve_expr.calc_data(portfolio_daily)
portfolio_curves.tail(8)


date,c02,c05,equal_weight
date,f64,f64,f64
2024-01-24,-563.319885,2097.259888,766.970001
2024-01-25,-563.319885,2113.359924,775.02002
2024-01-26,-571.319885,2136.039917,782.360016
2024-01-27,-571.319885,2135.899902,782.290009
2024-01-29,-571.319885,2177.519897,803.100006
2024-01-30,-574.319885,2227.759857,826.719986
2024-01-31,-574.319885,2155.599884,790.639999
2024-02-01,-587.319885,2165.039886,788.860001


In [59]:
equal_runtime = col("date", *strategy_names, "equal_weight").monitor(
    "single_and_equal_weight",
    show_axis_label=True,
).line().runtime()

equal_runtime.plot(portfolio_curves, open_in_jupyter=True, auto_open=False, height=480)


## 5. convex_sr：用 Sharpe 目标求组合权重

`convex_sr` 的输入是一组收益或 PnL 列，输出每一列对应的权重。这里用：

```python
col("c02", "c05").opt.convex_sr(0.0, 1.0)
```

参数含义：

| 参数 | 含义 |
| --- | --- |
| `lower_bound=0.0` | 每个策略权重不能低于 0，即 long-only |
| `up_bound=1.0` | 每个策略权重不能高于 1 |
| 权重和 | 底层约束为权重和等于 1 |
| 目标 | 在样本内最大化 Sharpe 近似目标 |

它不是“未来一定更好”的按钮。它只是根据历史日 PnL 的均值、波动和协方差，求一个样本内更优的线性组合。真实使用时通常要做滚动窗口优化，比如只用过去 3 个月训练权重，再用未来 1 周或 1 个月验证。


In [60]:
weights = col(*strategy_names).opt.convex_sr(0.0, 1.0).calc_data(portfolio_daily)
weights


c02,c05
f64,f64
1.1102e-16,1.0


In [61]:
weight_values = {name: float(weights[name][0]) for name in strategy_names}
weight_table = pl.DataFrame(
    {
        "strategy": strategy_names,
        "weight": [weight_values[name] for name in strategy_names],
    }
)
weight_table


strategy,weight
str,f64
"""c02""",1.1102e-16
"""c05""",1.0


## 6. convex_sr 组合曲线

得到权重后，组合每日 PnL 是：

```text
convex_daily_pnl = w0 * strategy_0_daily_pnl + w1 * strategy_1_daily_pnl
```

然后再对每日组合 PnL 做累计和，得到组合净值曲线。下面把等权组合和 convex_sr 组合放在同一张图里比较。


In [62]:
convex_daily_expr = sum(
    col(name) * col.lit(weight_values[name])
    for name in strategy_names
).alias("convex_sr")

compare_curve_expr = col(
    "date",
    equal_weight_expr.sum().expanding().alias("equal_weight"),
    convex_daily_expr.sum().expanding().alias("convex_sr"),
)

compare_curves = compare_curve_expr.calc_data(portfolio_daily)
compare_curves.tail(10)


date,equal_weight,convex_sr
date,f64,f64
2024-01-22,795.279984,2153.799866
2024-01-23,816.339996,2195.999878
2024-01-24,766.970001,2097.259888
2024-01-25,775.02002,2113.359924
2024-01-26,782.360016,2136.039917
2024-01-27,782.290009,2135.899902
2024-01-29,803.100006,2177.519897
2024-01-30,826.719986,2227.759857
2024-01-31,790.639999,2155.599884


In [63]:
convex_runtime = col("date", "equal_weight", "convex_sr").monitor(
    "equal_vs_convex_sr",
    show_axis_label=True,
).line().runtime()

convex_runtime.plot(compare_curves, open_in_jupyter=True, auto_open=False, height=460)


## 7. 补充：组合收益统计

曲线只能告诉我们“看起来怎么样”，统计表能把收益、波动、回撤、偏度、峰度等指标列出来。`bt.returns_stats()` 的输入是 `date + 每期收益/PnL`。这里为了演示，直接把每日 PnL 当作每期收益序列来统计。


In [64]:
portfolio_returns = col(
    "date",
    equal_weight_expr,
    convex_daily_expr,
).calc_data(portfolio_daily)

stats_equal = col("date", "equal_weight").bt.returns_stats(periods_per_year=252).calc_data(portfolio_returns)
stats_convex = col("date", "convex_sr").bt.returns_stats(periods_per_year=252).calc_data(portfolio_returns)

stats_preview = (
    stats_equal
    .rename({"value": "equal_weight", "value_float": "equal_weight_float"})
    .join(
        stats_convex.rename({"value": "convex_sr", "value_float": "convex_sr_float"}),
        on="metric",
        how="inner",
    )
    .select("metric", "equal_weight", "convex_sr")
)

stats_preview.head(18)


metric,equal_weight,convex_sr
str,str,str
"""Start Index""","""2022-07-02""","""2022-07-02"""
"""End Index""","""2024-02-01""","""2024-02-01"""
"""Total Duration""","""579 days, 0:00:00""","""579 days, 0:00:00"""
"""Total Return [%]""",null,null
"""Benchmark Return [%]""",null,null
"""Annualized Return [%]""",null,null
"""Annualized Volatility [%]""","""75822.66090854956""","""149967.63236737464"""
"""Max Drawdown [%]""","""849025626070.9075""","""inf"""
"""Max Drawdown Duration""","""0:00:00""","""7 days, 0:00:00"""


## 8. 小结

这条 portfolio 管线的核心逻辑是：

```text
策略表达式 -> 信号 -> 持仓 -> 单策略 PnL -> 日 PnL -> 组合权重 -> 组合曲线
```

qust 的优势在这里比较明显：

1. 策略、持仓、回测、分组汇总、组合优化、画图都是表达式或表达式 runtime；
2. 单策略和组合策略使用同一套数据口径，不需要到处复制 pandas 代码；
3. `over("ticker")` 把每个品种的状态隔离开，避免多品种回测互相串状态；
4. `convex_sr` 可以直接对多列策略收益求权重，方便批量策略筛选和组合研究；
5. notebook 输出的是真实 monitor iframe，可以继续缩放、拖动和检查曲线。

真实投研里，下一步通常是把 `convex_sr` 放进滚动训练框架：例如过去 60 个交易日训练权重，未来 5 个交易日应用权重，然后不断向前滚动。这样才更接近实际可交易的组合构建过程。
